In [8]:
import numpy as np

f0 = 429e12
h_const = 6.626e-34
G = 6.67e-11
M0 = 2e30
c = 3e8

In [127]:
def compute_complex_response(
    positions,
    arms,          # here: photon propagation directions n̂
    f_gw,          
    A_plus,
    A_cross,
    theta,
    phi_sky,
    psi=0
):
    # GW propagation vector
    k = -np.array([
        np.sin(theta) * np.cos(phi_sky),
        np.sin(theta) * np.sin(phi_sky),
        np.cos(theta)
    ])

    def make_perp_basis(k):
        k = k / np.linalg.norm(k)
        if np.abs(k[2]) < 0.99:
            tmp = np.array([0,0,1])
        else:
            tmp = np.array([1,0,0])
        ex = np.cross(tmp, k)
        ex /= np.linalg.norm(ex)
        ey = np.cross(k, ex)
        return ex, ey

    ex, ey = make_perp_basis(k)
    # Polarization tensors
    e_plus = np.outer(ex, ex) - np.outer(ey, ey)
    e_cross = np.outer(ex, ey) + np.outer(ey, ex)

    # Rotate by polarization angle psi
    e_plus_rot  = e_plus * np.cos(2 * psi) + e_cross * np.sin(2 * psi)
    e_cross_rot = -e_plus * np.sin(2 * psi) + e_cross * np.cos(2 * psi)

    # --- CLOCK ANTENNA GEOMETRY ---
    # n̂ = photon propagation direction
    nk = np.einsum("di,i->d", arms, k)
    denom = 1.0 - nk

    # Geometry-only detector tensor
    D = 0.5 * np.einsum("di,dj->dij", arms, arms) / denom[:, None, None]

    # Antenna factors
    F_plus  = np.einsum("dij,ij->d", D, e_plus_rot)
    F_cross = np.einsum("dij,ij->d", D, e_cross_rot)

    # Keep original phase convention
    tau = np.dot(positions, k) / c

    # Pure antenna-pattern response (no transfer function)
    S = (F_plus * A_plus + F_cross * A_cross) * np.exp(-1j * 2 * np.pi * f_gw * tau)
    
    

    return S, nk

def compute_fd_complex_response(
    positions,
    arm_lengths,
    arms,          # here: photon propagation directions n̂
    f_gw,          
    A_plus,
    A_cross,
    theta,
    phi_sky,
    psi=0
):
    arm_lengths = np.array(arm_lengths)
    # GW propagation vector
    k = -np.array([
        np.sin(theta) * np.cos(phi_sky),
        np.sin(theta) * np.sin(phi_sky),
        np.cos(theta)
    ])

    def make_perp_basis(k):
        k = k / np.linalg.norm(k)
        if np.abs(k[2]) < 0.99:
            tmp = np.array([0,0,1])
        else:
            tmp = np.array([1,0,0])
        ex = np.cross(tmp, k)
        ex /= np.linalg.norm(ex)
        ey = np.cross(k, ex)
        return ex, ey

    ex, ey = make_perp_basis(k)

    # Polarization tensors
    e_plus = np.outer(ex, ex) - np.outer(ey, ey)
    e_cross = np.outer(ex, ey) + np.outer(ey, ex)

    # Rotate by polarization angle psi
    e_plus_rot  = e_plus * np.cos(2 * psi) + e_cross * np.sin(2 * psi)
    e_cross_rot = -e_plus * np.sin(2 * psi) + e_cross * np.cos(2 * psi)

    # --- CLOCK ANTENNA GEOMETRY ---
    # n̂ = photon propagation direction
    nk = np.einsum("di,i->d", arms, k)
    denom = 1.0 - nk

    # Geometry-only detector tensor
    D = 0.5 * np.einsum("di,dj->dij", arms, arms) / denom[:, None, None]

    # Antenna factors
    F_plus  = np.einsum("dij,ij->d", D, e_plus_rot)
    F_cross = np.einsum("dij,ij->d", D, e_cross_rot)

    # Keep original phase convention
    tau = np.dot(positions, k) / c
    
    phase_delay = 2 * np.pi * f_gw * arm_lengths * denom / c
    transfer = 1.0 - np.exp(-1j * phase_delay)
    
    # Pure antenna-pattern response (no transfer function)
    S = (F_plus * A_plus + F_cross * A_cross) * np.exp(-1j * 2 * np.pi * f_gw * tau) * transfer
    
    

    return S, nk

In [128]:
##Source Setup

M_chirp  = 1000 ##In Solar Masses (Start at 500, go up to 2000)


##Natural_Polarization_Frame
inc_angle =np.pi/3

A_plus  = np.cos(inc_angle)
A_cross = (1+np.cos(inc_angle)**2)/2

##Rotating into detector frame 
psi  = np.pi/3 


##Sky Location
theta_sky  = -0.4
phi_sky  = -0.7




##Measurement Cycles
f_gw1 = 1/100 ##Keep this in the 10s of mHz 
## Make a few data sets where the distinction 

#Network Setup
d1 = 1e10

n_ensembles = 5
theta_1 = 0

r = 1.46e11 
positions = np.array([[r*np.cos(theta_1),r*np.sin(theta_1),0],[r*np.cos(theta_1),r*np.sin(theta_1),0]])
arms = np.array([[np.sin(theta_1),np.cos(theta_1),0],[-np.sin(theta_1),-np.cos(theta_1),0]])
arm_lengths =np.array([1e10,1e10])



N_cycles = [100,200,500,1000,2000]




In [125]:
compute_complex_response(positions, arms, f_gw1 ,A_plus, A_cross, theta = theta_sky, phi_sky = phi_sky,psi = psi)

(array([0.11152873+0.03662145j, 0.18622669+0.06114919j]),
 array([-0.25087018,  0.25087018]))

In [126]:
compute_fd_complex_response(positions,arm_lengths, arms, f_gw1 ,A_plus, A_cross, theta_sky, phi_sky,psi)

(array([0.18996403+0.12395812j, 0.12473821+0.24726413j]),
 array([-0.25087018,  0.25087018]))